In [9]:
import pandas as pd
import openai
import time
import json
import re
import os

# --- 1. 配置 API ---
API_KEY = "sk-e47f0c7274c74000b66cc6efd4f12501" 
BASE_URL = "https://dashscope.aliyuncs.com/compatible-mode/v1" 
client = openai.OpenAI(api_key=API_KEY, base_url=BASE_URL)

# --- 2. Prompt (保持不变) ---
SYSTEM_PROMPT = """
你是一个资深的反诈数据分析师。请严格按照以下规则分析文本，并仅输出合法的 JSON 对象。

### 第一步：判断来源类型 (source_type)
- **案件报道/通报**：包含"记者、警方、法院、新闻、通讯员"等字眼，属于第三方描述。
- **原始诈骗信息**：骗子发的短信/聊天，或受害者/受害者家属的直接描述。

### 第二步：归类到 11 大高发类型 (fraud_category)
必须严格从以下列表中**单选**一个最贴切的（仅输出文字，不带序号）：
["刷单返利类诈骗", "虚假网络投资理财类诈骗", "虚假购物、服务类诈骗", "冒充电商物流客服类诈骗", "虚假贷款类诈骗", "虚假征信类诈骗", "冒充领导、熟人类诈骗", "冒充公检法及政府机关类诈骗", "网络婚恋、交友类（杀猪盘）诈骗", "网络游戏虚假交易类诈骗", "其他"]

⚠️ 判定规则：
1. **虚假贷款类诈骗**：以“无抵押、低息、秒到账”为诱饵，诱导下载APP、缴纳保证金/解冻费。
2. **虚假征信类诈骗**：以“注销校园贷、调整利率、消除不良征信”为由，诱导转账或借贷。
3. 若信息不足或完全不属于上述10类，必须填"其他"。

### 第三步：新类型标准化命名 (new_category_name)
- 仅当 fraud_category 为 "其他" 时填写简练名词（如"AI换脸诈骗"）。
- 常规诈骗必须严格填 "无"。

### 第四步：受害者画像分析
1. **age_group**: 仅限 ["青少年", "青年", "中年", "老年", "未知"]
2. **gender**: 仅限 ["男", "女", "未知"]
3. **education_level**: 仅限 ["小学及以下", "初中", "高中/中专", "大专", "本科", "硕士及以上", "未知"]
4. **occupation**: 必须严格映射至以下标准库（仅输出主标签）：
   ["学生", "教师", "医护人员", "公务员/事业单位人员", "企业职员", "个体户/经商", "务工人员", "退休人员", "家庭主妇/主夫", "待业/无业", "其他"]
5. **economic_status**: 仅限 ["低收入", "中等收入", "高收入", "贫困", "富有", "未知"]

### 第五步：诈骗行为与损失提取
1. **contact_channel** (接触渠道): 
   - 常见渠道：["微信", "QQ", "抖音", "快手", "短信", "电话", "网络平台", "陌生链接", "招聘平台"]
   - **境外APP细化规则**：如果文中明确提及具体境外APP名称，**必须直接输出该名称**（例如："Telegram", "WhatsApp", "Line", "Viber", "Signal", "Tinder", "Facebook", "Instagram", "Twitter", "Skype", "钉钉"等）。
   - 若仅提及是境外软件但未指名，或泛指此类软件，填 "境外APP"。
   - 若不属于以上任何情况，填 "其他" 或 "未知"。
2. **loss_amount** (损失金额): 提取明确金额，统一转为纯数字+“元”（例：“1.2万”→“12000元”）。未提及严格填“未提及”。

### 第六步：行为特征分析
1. **fund_behavior** (资金行为): 仅限 ["小额试水", "大额投入", "多次转账", "犹豫不决", "快速响应", "未知"]
2. **decision_trait** (决策特征): 仅限 ["冲动决定", "多方咨询", "谨慎核实", "盲目相信", "情绪化决策", "未知"]
3. **post_event_behavior** (事后行为): 仅限 ["立即报警", "尝试挽回", "隐瞒不报", "反思总结", "告知亲友", "未知"]

### 📋 严格输出要求
- 仅输出合法 JSON，**严禁包含 Markdown 标记、解释性文字或多余符号**。
- JSON 键名必须与下方示例完全一致：
{
    "source_type": "",
    "fraud_category": "",
    "new_category_name": "",
    "age_group": "",
    "gender": "",
    "education_level": "",
    "occupation": "",
    "economic_status": "",
    "contact_channel": "",
    "loss_amount": "",
    "fund_behavior": "",
    "decision_trait": "",
    "post_event_behavior": ""
}
"""

def call_ai(text):
    try:
        response = client.chat.completions.create(
            model="qwen-plus",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"请分析文本：\n{text}"}
            ],
            temperature=0.1,
            response_format={"type": "json_object"}
        )
        content = response.choices[0].message.content
        # 清洗 Markdown 格式
        content = re.sub(r'^```(json)?\s*', '', content, flags=re.IGNORECASE).strip()
        content = re.sub(r'\s*```$', '', content, flags=re.IGNORECASE).strip()
        return content
    except Exception as e:
        return json.dumps({"error": str(e)})

def parse_json(safe_str):
    try:
        # 修复可能的 JSON 格式错误（如末尾逗号）
        safe_str = re.sub(r',(\s*[}\]])', r'\1', safe_str)
        return json.loads(safe_str)
    except json.JSONDecodeError:
        return {"error": "json_parse_failed"}

def standardize_amount(val):
    # 增加类型检查，防止传入非字符串
    if not isinstance(val, str):
        val = str(val)
    if not val or val == "未提及":
        return "未提及"
    val = val.replace(",", "").replace(" ", "")
    if not val.endswith("元"):
        clean_val = re.sub(r'[^\d.]', '', val)
        if clean_val:
            try:
                num = float(clean_val)
                val = f"{int(num)}元" if num.is_integer() else f"{num}元"
            except:
                val = f"{val}元"
        else:
            val = "未提及"
    return val

# --- 3. 主程序 (修复了数组比较错误) ---
def main():
    input_file = 'post提取-2.0.xlsx'
    output_file = 'post提取-2.0_已提取.xlsx'
    BATCH_SIZE = 10       
    
    NEW_COLS = [
        "source_type", "fraud_category", "new_category_name", 
        "age_group", "gender", "education_level", "occupation", "economic_status",
        "contact_channel", "loss_amount",
        "fund_behavior", "decision_trait", "post_event_behavior"
    ]
    
    try:
        if not os.path.exists(input_file):
            print(f"❌ 错误：找不到文件 {input_file}")
            return

        df = pd.read_excel(input_file, engine='openpyxl')
        
        if 'clean_content' not in df.columns:
            print(f"❌ 错误：Excel 中未找到 'clean_content' 列。可用列: {df.columns.tolist()}")
            return
            
        df_test = df.copy()
        for col in NEW_COLS:
            df_test[col] = None
            
        total_rows = len(df_test)
        print(f"📊 开始处理，总数据量: {total_rows} 条")
        
        processed_count = 0

        for idx in range(total_rows):
            text = str(df_test.loc[idx, 'clean_content'])
            if pd.isna(text) or text.strip() in ['', 'nan', 'None']:
                text = ""
            
            raw_json = call_ai(text)
            data = parse_json(raw_json)
            
            if "error" not in data:
                for col in NEW_COLS:
                    val = data.get(col, "")
                    
                    # --- 修复核心：强制转为字符串并安全判断 ---
                    # 1. 强制转为字符串，防止列表或数字类型导致的比较错误
                    if isinstance(val, list):
                        val = val[0] if val else "" # 如果是列表，取第一个元素
                    val_str = str(val).strip()
                    
                    # 2. 使用安全的字符串判断
                    if val_str == "" or val_str.lower() == "null" or val_str == "nan":
                        if col == "new_category_name": 
                            val_str = "无"
                        elif col == "loss_amount": 
                            val_str = "未提及"
                        else: 
                            val_str = "未知"
                    
                    # 3. 特殊处理金额
                    if col == "loss_amount":
                        val_str = standardize_amount(val_str)
                        
                    df_test.loc[idx, col] = val_str
                status = "✅"
            else:
                # 错误处理
                for col in NEW_COLS:
                    df_test.loc[idx, col] = "无" if col == "new_category_name" else ("未提及" if col == "loss_amount" else "未知")
                status = "❌"
            
            processed_count += 1
            
            # 每50条报告一次进度
            if processed_count % 50 == 0:
                progress_percent = (processed_count / total_rows) * 100
                print(f"📌 进度: 已处理 {processed_count} 条 ({progress_percent:.1f}%)")
            
            # 每1000条保存一次文件
            if processed_count % 1000 == 0:
                df_test.to_excel(output_file, index=False, engine='openpyxl')
                print(f"   💾 已自动保存文件...")
            
            time.sleep(0.4)

        # --- 最终保存 ---
        df_test.to_excel(output_file, index=False, engine='openpyxl')
        print(f"\n🎉 全部提取完成！最终文件已保存至: 📄 {output_file}")
        
        df_test.to_csv('full_extract_result.csv', encoding='utf-8-sig', index=False)

        # 最终统计
        print("\n📊 处理完成统计:")
        print("- 诈骗类型分布:")
        print(df_test['fraud_category'].value_counts())

    except Exception as e:
        print(f"❌ 程序崩溃: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

📊 开始处理，总数据量: 1993 条
📌 进度: 已处理 50 条 (2.5%)
📌 进度: 已处理 100 条 (5.0%)
📌 进度: 已处理 150 条 (7.5%)
📌 进度: 已处理 200 条 (10.0%)
📌 进度: 已处理 250 条 (12.5%)
📌 进度: 已处理 300 条 (15.1%)
📌 进度: 已处理 350 条 (17.6%)
📌 进度: 已处理 400 条 (20.1%)
📌 进度: 已处理 450 条 (22.6%)
📌 进度: 已处理 500 条 (25.1%)
📌 进度: 已处理 550 条 (27.6%)
📌 进度: 已处理 600 条 (30.1%)
📌 进度: 已处理 650 条 (32.6%)
📌 进度: 已处理 700 条 (35.1%)
📌 进度: 已处理 750 条 (37.6%)
📌 进度: 已处理 800 条 (40.1%)
📌 进度: 已处理 850 条 (42.6%)
📌 进度: 已处理 900 条 (45.2%)
📌 进度: 已处理 950 条 (47.7%)
📌 进度: 已处理 1000 条 (50.2%)
   💾 已自动保存文件...
📌 进度: 已处理 1050 条 (52.7%)
📌 进度: 已处理 1100 条 (55.2%)
📌 进度: 已处理 1150 条 (57.7%)
📌 进度: 已处理 1200 条 (60.2%)
📌 进度: 已处理 1250 条 (62.7%)
📌 进度: 已处理 1300 条 (65.2%)
📌 进度: 已处理 1350 条 (67.7%)
📌 进度: 已处理 1400 条 (70.2%)
📌 进度: 已处理 1450 条 (72.8%)
📌 进度: 已处理 1500 条 (75.3%)
📌 进度: 已处理 1550 条 (77.8%)
📌 进度: 已处理 1600 条 (80.3%)
📌 进度: 已处理 1650 条 (82.8%)
📌 进度: 已处理 1700 条 (85.3%)
📌 进度: 已处理 1750 条 (87.8%)
📌 进度: 已处理 1800 条 (90.3%)
📌 进度: 已处理 1850 条 (92.8%)
📌 进度: 已处理 1900 条 (95.3%)
📌 进度: 已处理 1950 条 (97.8%)

🎉 全部提取完成！最终